# 11 - 结构化输出

本 Notebook 学习 LangChain 的结构化输出功能，让 Agent 按照指定的格式返回结果。

## 学习目标

- ✅ 理解为什么需要结构化输出
- ✅ 掌握最简单的用法（传入 Pydantic 模型）
- ✅ 了解与工具共存的结构化输出
- ✅ 学会处理复杂嵌套结构
- ✅ 掌握 `with_structured_output()` 方法
- ✅ 理解三种输出策略的区别

---

## 为什么需要结构化输出

**术语：结构化输出（Structured Output）** — 让 Agent 按照指定的格式返回结果，方便程序直接使用，而不是返回自由文本。

假设你需要从一段用户描述中提取姓名、年龄和职业：

| 方式 | 输出格式 | 后续处理 |
| --- | --- | --- |
| 普通回复 | `"张三今年28岁，是一名工程师"` | 需要正则或再次调用模型来解析 |
| 结构化输出 | `{name: "张三", age: 28, job: "工程师"}` | 直接作为 Python 对象使用 |

**结构化输出的优势：**
- 省去了"从文本中解析数据"这一步
- AI 的输出可以直接被程序使用
- 类型安全，避免运行时错误

---

## 最简单的用法——传入 Pydantic 模型

**术语：Pydantic 模型** — Python 的数据验证库，用 `BaseModel` 定义数据结构，提供类型检查和验证。

将 Pydantic 模型传给 `response_format` 参数即可：

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_learning.llm import get_llm

# 定义期望的输出结构
class CourseInfo(BaseModel):
    """菜鸟教程 RUNOOB 课程提取结果"""
    course_name: str = Field(description="课程名称")
    difficulty: str = Field(description="难度：入门/进阶/高级")
    estimated_hours: int = Field(description="预计学习时长（小时）")
    is_free: bool = Field(description="是否免费")

# 使用项目封装的 LLM
model = get_llm()

# 创建 Agent，传入 Pydantic 模型到 response_format
agent = create_agent(
    model=model,
    response_format=CourseInfo,  # 传入 Pydantic 模型
    system_prompt="你是菜鸟教程 RUNOOB 的课程助手，从用户描述中提取课程信息。",
)

# 用户输入一段非结构化的描述
result = agent.invoke({
    "messages": [HumanMessage(
        content="我最近在学习 Python3 基础教程，是入门级别的，大概要学 20 个小时，而且是完全免费的"
    )]
})

In [ ]:
# 从 structured_response 获取结构化结果
if "structured_response" in result:
    course = result["structured_response"]
    print(f"课程名: {course.course_name}")
    print(f"难度: {course.difficulty}")
    print(f"预计时长: {course.estimated_hours} 小时")
    print(f"免费: {'是' if course.is_free else '否'}")
    print(f"对象类型: {type(course)}")

**术语：structured_response** — Agent 返回结果中的结构化输出字段，包含符合 Schema 的数据。

> 返回的 `structured_response` 是 Pydantic 模型实例，而不是普通字典。这意味着你可以使用 `.course_name` 等属性访问，IDE 也能提供自动补全。

---

## 与工具共存的 Structured Output

**术语：工具（Tool）** — Agent 可以调用的外部函数，用于获取实时信息或执行操作。

`response_format` 和 `tools` 可以同时使用——Agent 在需要时调用工具，最终输出结构化数据：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_learning.llm import get_llm

@tool
def search_course(keyword: str) -> str:
    """在菜鸟教程 RUNOOB 搜索课程信息"""
    courses = {
        "python": "Python3 基础教程 | 入门 | 免费 | 30章 | 约20小时",
        "java": "Java 基础教程 | 入门 | 免费 | 35章 | 约25小时",
        "数据分析": "Python 数据分析 | 进阶 | 会员 | 25章 | 约30小时",
    }
    return courses.get(keyword.lower(), f"未找到 '{keyword}' 相关课程")

class CourseRecommendation(BaseModel):
    """课程推荐结果"""
    course_name: str = Field(description="推荐课程名称")
    reason: str = Field(description="推荐理由")
    difficulty: str = Field(description="难度：入门/进阶/高级")

model = get_llm()

# 同时使用 tools 和 response_format
agent = create_agent(
    model=model,
    tools=[search_course],
    response_format=CourseRecommendation,
    system_prompt="你是菜鸟教程 RUNOOB 的课程顾问。先查询课程再给出推荐。",
)

result = agent.invoke({
    "messages": [HumanMessage(content="我想学 Python，有什么推荐？")]
})

In [ ]:
rec = result["structured_response"]
print(f"推荐课程: {rec.course_name}")
print(f"推荐理由: {rec.reason}")
print(f"难度: {rec.difficulty}")

# 查看完整过程
print("\n=== 执行过程 ===")
for msg in result["messages"]:
    if msg.type == "tool":
        print(f"  调用 {msg.name}: {msg.content}")

**执行流程：**

```mermaid
sequenceDiagram
    participant U as 用户
    participant A as Agent
    participant T as 工具
    
    U->>A: "我想学 Python"
    A->>T: search_course("python")
    T-->>A: 返回课程信息
    A->>A: 生成结构化推荐
    A-->>U: CourseRecommendation 对象
```

---

## 复杂嵌套结构

Pydantic 支持嵌套、列表、枚举等复杂结构：

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_learning.llm import get_llm

class Topic(BaseModel):
    """知识点"""
    name: str = Field(description="知识点名称")
    order: int = Field(description="学习顺序，从 1 开始")
    minutes: int = Field(description="建议学习分钟数")

class LearningPlan(BaseModel):
    """学习计划"""
    goal: str = Field(description="学习目标概述")
    level: Literal["入门", "进阶", "高级"] = Field(description="难度级别")
    total_hours: float = Field(description="总时长（小时）")
    topics: list[Topic] = Field(description="知识点列表")

model = get_llm()

agent = create_agent(
    model=model,
    response_format=LearningPlan,
    system_prompt="你是菜鸟教程 RUNOOB 的学习规划师。",
)

result = agent.invoke({
    "messages": [HumanMessage(
        content="帮我制定一个 Python 入门学习计划，总时长控制在 10 小时以内"
    )]
})

In [ ]:
plan = result["structured_response"]
print(f"目标: {plan.goal}")
print(f"难度: {plan.level}")
print(f"总时长: {plan.total_hours} 小时")
print(f"\n知识点列表 ({len(plan.topics)} 个):")
for topic in plan.topics:
    print(f"  {topic.order}. {topic.name} ({topic.minutes}分钟)")

**术语：Literal** — Python 类型注解，限制字段只能是预定义的几个值之一（如"入门"/"进阶"/"高级"）。

**术语：嵌套结构** — 在一个 Pydantic 模型中引用另一个模型，形成层级关系（如 LearningPlan 包含多个 Topic）。

---

## 从消息中获取结构化输出

**术语：with_structured_output()** — 模型方法，直接从文本中提取结构化信息，不需要 Agent。

如果你的场景是"信息提取"而非"多步骤推理"，直接用 `with_structured_output()` 更简洁高效：

In [ ]:
from pydantic import BaseModel, Field
from langchain_learning.llm import get_llm

class SentimentResult(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="积极/消极/中性")
    score: float = Field(description="情感强度 0~1")
    keywords: list[str] = Field(description="关键情感词")

# 获取 LLM 实例
model = get_llm()

# 直接在模型上使用 with_structured_output()
# 不需要 Agent
structured_model = model.with_structured_output(SentimentResult)

texts = [
    "菜鸟教程 RUNOOB 真的太好用了，强烈推荐！",
    "这个教程内容太少了，不太值。",
    "今天天气不错。",
]

for text in texts:
    result = structured_model.invoke(text)
    print(f"文本: {text[:30]}...")
    print(f"  情感: {result.sentiment}, 强度: {result.score}, 关键词: {result.keywords}")

**使用场景对比：**

| 场景 | 推荐方式 | 原因 |
| --- | --- | --- |
| 信息提取 | `with_structured_output()` | 简洁高效，不需要 Agent |
| 多步骤推理 | `create_agent()` + `response_format` | 可以调用工具、多轮对话 |
| 需要工具调用 | `create_agent()` + `response_format` | Agent 可以调用工具获取信息 |

---

## 三种输出策略

LangChain 提供了三种结构化输出策略，理解它们的区别和工作原理，能帮助你在不同场景下做出最佳选择。

### 三种策略概述

| 策略 | 原理 | 模型支持 | 响应速度 |
| --- | --- | --- | --- |
| **ToolStrategy** | 将 Schema 伪装成工具，模型"调用"这个工具来输出结构化数据 | 所有支持 function calling 的模型 | 较慢（多一次工具调用） |
| **ProviderStrategy** | 使用模型原生的结构化输出能力（如 OpenAI 的 response_format） | 部分模型（GPT-4o+、Claude 3+等） | 较快（直接输出） |
| **AutoStrategy** | 自动检测模型能力，选择最佳策略 | 自动适配 | 自动选择最优 |

### ToolStrategy——工具调用模式

**术语：ToolStrategy** — 将 Schema 转换为一个"假工具"，模型通过调用这个工具来输出结构化数据。这是兼容性最好的方式。

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
from langchain_learning.llm import get_llm

class WeatherReport(BaseModel):
    """天气报告"""
    city: str = Field(description="城市名称")
    temperature: float = Field(description="温度（摄氏度）")
    condition: str = Field(description="天气状况")
    humidity: int = Field(description="湿度百分比")

model = get_llm()

# 显式指定使用 ToolStrategy
agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=WeatherReport),
    system_prompt="你是天气助手，根据用户描述生成结构化天气报告。",
)

result = agent.invoke({
    "messages": [HumanMessage(content="杭州今天晴天，温度25度，湿度60%")]
})

In [ ]:
report = result["structured_response"]
print(f"城市: {report.city}")
print(f"温度: {report.temperature}°C")
print(f"状况: {report.condition}")
print(f"湿度: {report.humidity}%")

# 查看执行过程——可以发现多了一条工具调用的消息
print(f"\n消息总数: {len(result['messages'])}")
for msg in result["messages"]:
    print(f"  [{msg.type}]", end="")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f" 调用: {[tc['name'] for tc in msg.tool_calls]}")
    elif msg.type == "tool":
        print(f" {msg.content[:60]}...")
    else:
        print(f" {str(msg.content)[:60]}...")

**ToolStrategy 工作流程：**

```mermaid
sequenceDiagram
    participant U as 用户
    participant A as Agent
    participant M as 模型
    participant T as 假工具
    
    U->>A: 输入描述
    A->>M: 调用模型
    M->>T: 调用 WeatherReport 工具
    T-->>M: 返回结构化数据
    M-->>A: AIMessage + tool_calls
    A-->>U: structured_response
```

可以看到 ToolStrategy 多了一个工具调用步骤（调用名为 WeatherReport 的"假工具"），然后才有结构化输出。

**handle_errors——错误重试**

ToolStrategy 支持在结构化输出出错时自动重试：

In [ ]:
from langchain.agents.structured_output import ToolStrategy

# handle_errors=True：输出格式错误时，将错误信息反馈给模型重试
strategy_with_retry = ToolStrategy(
    schema=WeatherReport,
    handle_errors=True,  # 默认 False
)

# handle_errors 也可以是一个自定义错误消息模板
strategy_custom_error = ToolStrategy(
    schema=WeatherReport,
    handle_errors="格式有误，请按 {error} 修正后重新输出",
)

### ProviderStrategy——原生结构化输出

**术语：ProviderStrategy** — 使用模型提供商的原生能力（如 OpenAI 的 response_format 参数）。不是所有模型都支持。

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from langchain_learning.llm import get_llm

class CourseInfo(BaseModel):
    """课程信息"""
    name: str = Field(description="课程名称")
    level: str = Field(description="难度：入门/进阶/高级")
    price: str = Field(description="价格信息")

model = get_llm()

# 显式指定 ProviderStrategy
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(schema=CourseInfo),
    system_prompt="你是菜鸟教程 RUNOOB 的课程助手。",
)

result = agent.invoke({
    "messages": [HumanMessage(content="Python3 基础教程是入门级的免费课程")]
})

In [ ]:
course = result["structured_response"]
print(f"课程: {course.name}")
print(f"难度: {course.level}")
print(f"价格: {course.price}")
print(f"\n消息数: {len(result['messages'])}")  # 比 ToolStrategy 少

与 ToolStrategy 相比，ProviderStrategy 的消息数更少（2 条 vs 4 条），因为它不需要额外的工具调用步骤。

> ProviderStrategy 目前主要被 OpenAI 的 GPT-4o 及以上和 Claude 3 及以上支持。如果模型不支持，LangChain 会自动降级到 ToolStrategy。

### AutoStrategy——自动选择

这是最推荐的方式。传入 Pydantic 模型（而不是策略对象），LangChain 会自动选择最佳策略：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_learning.llm import get_llm

class Analysis(BaseModel):
    """分析结果"""
    summary: str = Field(description="一句话总结")
    score: int = Field(description="评分 1~10")
    pros: list[str] = Field(description="优点列表")
    cons: list[str] = Field(description="缺点列表")

model = get_llm()

# 直接传入 Pydantic 模型——LangChain 自动选择策略
agent = create_agent(
    model=model,
    response_format=Analysis,  # 直接传模型，自动选择策略
    system_prompt="你是课程评估专家，评估用户描述的课程。",
)

result = agent.invoke({
    "messages": [HumanMessage(
        content="菜鸟教程 RUNOOB 的 Python 课程：内容系统全面，实例丰富，而且完全免费。但视频教程较少，高级内容覆盖不够。"
    )]
})

In [ ]:
analysis = result["structured_response"]
print(f"总结: {analysis.summary}")
print(f"评分: {analysis.score}/10")
print(f"优点: {', '.join(analysis.pros)}")
print(f"缺点: {', '.join(analysis.cons)}")

### 三种策略选择指南

| 场景 | 推荐策略 | 原因 |
| --- | --- | --- |
| 不确定模型是否支持原生输出 | **AutoStrategy**（直接传 Pydantic） | 自动选择最优策略 |
| 需要兼容各种模型 | **ToolStrategy** | 所有支持 function calling 的模型都可用 |
| 追求极致性能 | **ProviderStrategy** | 跳过工具调用环节，速度更快 |
| 需要错误重试 | **ToolStrategy(handle_errors=True)** | 只有 ToolStrategy 支持 handle_errors |

> 大多数情况下，直接传入 Pydantic 模型（即使用 AutoStrategy）就够了。只有在需要错误重试或明确控制策略行为时，才需要显式指定 ToolStrategy 或 ProviderStrategy。

---

## 总结

本 Notebook 学习了 LangChain 的结构化输出功能：

**核心概念：**
- **结构化输出** — 让 Agent 按照指定格式返回结果
- **Pydantic 模型** — 定义输出结构，提供类型验证
- **structured_response** — Agent 返回的结构化数据字段

**使用方式：**
| 场景 | 方式 |
| --- | --- |
| 信息提取 | `model.with_structured_output(Schema)` |
| Agent 结构化输出 | `create_agent(response_format=Schema)` |
| Agent + 工具 + 结构化输出 | `create_agent(tools=[...], response_format=Schema)` |

**三种策略：**
| 策略 | 特点 | 适用场景 |
| --- | --- | --- |
| AutoStrategy | 自动选择最优策略 | 大多数情况（推荐） |
| ToolStrategy | 兼容性好，支持错误重试 | 需要兼容各种模型或错误重试 |
| ProviderStrategy | 性能最好 | 模型支持且追求极致性能 |

**下一步：** 继续学习 `12-memory.ipynb`，掌握对话记忆功能！